In [1]:
from utils.data import ProteinDataset
import torch as pt


data = pt.load(f'./data/pbond0_hbond0.pt')
datalib = ProteinDataset(data)
pdb2idx = [(data[2][i], i) for i in range(len(data[2]))] # pdb name -> idx
pdb2idx = dict(pdb2idx)

/tmp/ipykernel_30910/1951596569.py:5: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  data = pt.load(f'./data/pbond0_hbond0.pt')


In [ ]:
import torch as pt
import torch.nn as nn
import torch_geometric.nn as gnn


class ProteinGCN(nn.Module):
    def __init__(self, embed_dim:int=512, hidden_channels:int=256, out_channels:int=128, num_layers:int=3, num_edge_features:int=10):
        super().__init__()
        self.emb = nn.Embedding(num_embeddings=21, embedding_dim=embed_dim, padding_idx=0)
        # node_attr占一维
        self.gcn = gnn.GCN(in_channels=embed_dim+num_edge_features, hidden_channels=hidden_channels, 
                           num_layers=num_layers, out_channels=out_channels)
        self.shared = nn.Sequential(nn.Linear(4*out_channels, out_channels), nn.ReLU(),)
        self.tm_head = nn.Linear(out_channels, 1)
        self.seq_head = nn.Linear(out_channels, 1)

    def embed(self, seq_mask):
        seq, mask = seq_mask
        embedding = self.emb(seq) # [batch_size, seq_len, emb_dim]
        embedding = embedding * mask.unsqueeze(-1) # mask: [batch_size, seq_len, 1]
        return embedding

    def encode_protein(self, seq, mask, graph):
        x, edge_idx, edge_attr, batch, node2seq = graph.x, graph.edge_index, graph.edge_attr, graph.batch, graph.node2seq
        emb = self.embed((seq, mask))
        B, L, D = emb.shape
        emb_flat = emb.view(-1, D)
        flat_idx = batch * L + node2seq
        node_emb = emb_flat[flat_idx]
        x = pt.cat([node_emb, x], dim=-1)
        x = self.gcn(x, edge_idx, edge_attr=edge_attr, batch=batch)
        x = gnn.global_mean_pool(x, batch)
        return x        

    def forward(self, data, mode:str='pretraining'):
        if mode == 'pretraining':
            (seqs, masks, graphs), (inv_i, inv_j) = data
            prot_repr = self.encode_protein(seqs, masks, graphs)
            x_i = prot_repr[inv_i]
            x_j = prot_repr[inv_j]
            feature = pt.cat([x_i, x_j, x_i-x_j, x_i*x_j], dim=-1)
            shared = self.shared(feature)
            tm_score = self.tm_head(shared).squeeze(-1)
            seq_score = self.seq_head(shared).squeeze(-1)
            return tm_score, seq_score
        elif mode == 'finetuning':
            pass
        else:
            raise ValueError(f'Unknown mode: {mode}')

In [3]:
from sklearn.model_selection import train_test_split
import numpy as np
from torch.utils.data import DataLoader
from utils.data import ProteinPairDataset, pair_collate_fun


pair_dataset = ProteinPairDataset(datalib, './data/tmalign.out', pdb2idx)
batch_size = 512
loader = DataLoader(pair_dataset, batch_size=batch_size, shuffle=False, collate_fn=pair_collate_fun(datalib), num_workers=6)
gpu = 6
data_map = np.arange(len(pair_dataset), dtype=np.int64)
train_map, test_map = train_test_split(data_map, test_size=10240, random_state=42)
train_set = ProteinPairDataset(pair_dataset, mapping=train_map)
test_set = ProteinPairDataset(pair_dataset, mapping=test_map)
train_loader = DataLoader(train_set, batch_size=batch_size, shuffle=True, 
                          collate_fn=pair_collate_fun(datalib), drop_last=True, num_workers=6)
test_loader = DataLoader(test_set, batch_size=batch_size, shuffle=False, 
                         collate_fn=pair_collate_fun(datalib), num_workers=6)

In [ ]:
import logging
from tqdm import tqdm


lamda = 0.1
prot_model = ProteinGCN().to(gpu)
criterion = nn.SmoothL1Loss()
learning_rate = 1e-3
optimizer = pt.optim.AdamW(prot_model.parameters(), lr=learning_rate)
num_epochs = 10

for epoch in range(num_epochs):
    train_loss = []
    prot_model.train()
    for i, batch in enumerate(tqdm(train_loader, unit='batch')):
        prot, inv, score = batch
        prot = [x.to(gpu) for x in prot]
        inv = [x.to(gpu) for x in inv]
        score = score.to(gpu)
        output = prot_model((prot, inv))
        tm_score, seq_score = output
        tm_loss = criterion(tm_score, score[:, 0])
        seq_loss = criterion(seq_score, score[:, 1])
        loss = tm_loss + seq_loss
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        train_loss.append(loss.item())
        if (i+1) % 500 == 0:
            l = pt.tensor(train_loss).mean()
            print(f'Epoch [{epoch+1}/{num_epochs}], Train Loss: {l:.4f}')
            train_loss = []

  4%|▍         | 101/2352 [00:14<04:28,  8.39batch/s]

Epoch [1/10], Train Loss: 0.0123


  9%|▊         | 201/2352 [00:26<04:24,  8.12batch/s]

Epoch [1/10], Train Loss: 0.0033


 13%|█▎        | 301/2352 [00:38<04:11,  8.15batch/s]

...

 94%|█████████▎| 2201/2352 [04:22<00:18,  8.19batch/s]

Epoch [10/10], Train Loss: 0.0015


 98%|█████████▊| 2301/2352 [04:34<00:05,  8.55batch/s]

Epoch [10/10], Train Loss: 0.0015


100%|██████████| 2352/2352 [04:41<00:00,  8.36batch/s]


In [5]:
pt.cuda.empty_cache()